**This notebook shows how to localize with PixLoc, visualize different quantities, and generate interactive animations.**

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from pprint import pformat
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import HTML
import cv2

from pixloc.settings import DATA_PATH, LOC_PATH
from pixloc.localization import RetrievalLocalizer, PoseLocalizer, SimpleTracker
from pixloc.pixlib.geometry import Camera, Pose
from pixloc.visualization.viz_2d import (
    plot_images, plot_keypoints, add_text, features_to_RGB)
from pixloc.visualization.animation import (
    subsample_steps, VideoWriter, create_viz_dump, display_video)

In [ ]:
dataset = 'zedx_mini'
from pixloc.run_zedx_mini import default_paths, default_confs, pose_priors

print(f'default paths:\n{pformat(default_paths.asdict())}')
paths = default_paths.add_prefixes(DATA_PATH/dataset, LOC_PATH/dataset)

use_pose_priors = True
if use_pose_priors:
    conf = default_confs['from_poses']
    paths.hloc_logs = LOC_PATH / dataset / pose_priors
    print(f'conf:\n{pformat(conf)}')
    localizer = PoseLocalizer(paths, conf)
else:
    conf = default_confs['from_retrieval']
    conf['refinement']['do_pose_approximation'] = False
    print(f'conf:\n{pformat(conf)}')
    localizer = RetrievalLocalizer(paths, conf)

# Localize one query

In [ ]:
# name_q = np.random.choice(list(localizer.queries))  # pick a random query
name_q = '1778982327799078000.png' # or select one for ZEDX Mini camera

tracker = SimpleTracker(localizer.refiner)  # will hook & store the predictions
cam_q = Camera.from_colmap(localizer.queries[name_q])
ret = localizer.run_query(name_q, cam_q)
if ret['success']:
    print('Succeed to localize query ' + name_q)
else:
    print('Fail to localize query ' + name_q)
image_q = paths.query_images / name_q
img = cv2.imread(str(image_q))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plot_images([img], dpi=75)

# Visualize the CNN predictions

In [ ]:
ref_ids = ret['dbids'][:3]  # show 3 references at most
names_r = [localizer.model3d.dbs[i].name for i in ref_ids]
for name in names_r + [name_q]:
    image, features, weights = tracker.dense[name][1]
    features = features_to_RGB(features[1].numpy())[0]
    plot_images([image, features, weights[1]], cmaps='turbo',
                titles=[name, 'features', 'confidence'], dpi=50)

# Visualize the initial and final poses

In [ ]:
# Pick the first reference image for visualization
ref_id = ref_ids[0]
name_r = names_r[0]
ref = localizer.model3d.dbs[ref_id]
cam_r = Camera.from_colmap(localizer.model3d.cameras[ref.camera_id])
T_w2r = Pose.from_colmap(ref)

image_r = tracker.dense[name_r][1][0]
image_q = tracker.dense[name_q][1][0]
p3d = tracker.p3d
T_w2q_init = tracker.T[0]
T_w2q_final = tracker.T[-1]

# Project the 3D points
p2d_r, mask_r = cam_r.world2image(T_w2r * p3d)
p2d_f, mask_f = cam_q.world2image(T_w2q_final * p3d)
p2d_i, mask_i = cam_q.world2image(T_w2q_init * p3d)

plot_images([image_r, image_q], dpi=75)
plot_keypoints([p2d_r[mask_r], p2d_f[mask_f]], 'lime')
# plot_keypoints([None, p2d_i[mask_i]], 'red')
add_text(0, 'reference')
add_text(1, 'query')

# Plot the cost throughout the iterations

In [ ]:
costs = tracker.costs
fig, axes = plt.subplots(1, len(costs), figsize=(len(costs)*4.5, 4.5))
for i, (ax, cost) in enumerate(zip(axes, costs)):
    ax.plot(cost) if len(cost)>1 else ax.scatter(0., cost)
    ax.set_title(f'({i}) Scale {i//3} Level {i%3}')
    ax.grid()

# Generate a video

In [ ]:
T_w2q_all = torch.stack(tracker.T)
p2d_q_all, mask_q_all = cam_q.world2image(T_w2q_all * p3d)
keep = subsample_steps(T_w2q_all, p2d_q_all, mask_q_all, cam_q.size.numpy())
print(f'Keep {len(keep)}/{len(p2d_q_all)} optimization steps for visualization.')

In [ ]:
video = 'pixloc.mp4'
writer = VideoWriter('./tmp')
for i in tqdm(keep):
    plot_images([image_r, image_q], dpi=100, autoscale=False)
    plot_keypoints([p2d_r[mask_r], p2d_q_all[-1][mask_q_all[-1]]], 'lime')
    plot_keypoints([None, p2d_q_all[i][mask_q_all[i]]], 'red')
    add_text(0, 'reference')
    add_text(1, 'query')
    writer.add_frame()
writer.to_video(video, duration=4, crf=23)  # in seconds

display_video(video)

# Visualize in 3D!

In [ ]:
# the path relative to the viewer folder
viewer = Path('../viewer')
assets = viewer / 'dumps/sample'

# set the Y axis up for threejs
tfm = np.eye(3)

# Write a json dump to viewer/assets/sample.json
create_viz_dump(
    assets, paths, cam_q, name_q, T_w2q_all[keep],
    mask_q_all[keep], p2d_q_all[keep],
    ref_ids, localizer.model3d, tracker.p3d_ids, tfm=tfm)

with open(viewer / 'jupyter.html', 'r') as f:
    html = f.read()
html = html.replace('{__path__}', str(viewer))
html = html.replace('{__assets__}', str(assets.parent))
html = html.replace('{__height__}', '600px')
HTML(html)